# 11. Inference on the access measures

This notebook supplies the statistical inference for the relationships
estimated from the access measures: standard errors, probabilities, tests of
the difference between definitions, a measure of spatial dependence and
standard errors that allow for it.

The sections are these.

The coefficient on deprivation under every definition, within metropolitan
areas, with a covariance assembled across the whole system of equations. Since
every definition is fitted on the same tracts with the same characteristics,
the coefficients covary, and the difference between any two of them is reported
with its own standard error, t and probability.

Moran's statistic on the access measures and on the residuals of the models,
with each tract joined to the eight nearest tracts of its own metropolitan
area, and a pseudo probability from rearrangements carried out inside each
area. Standard errors are then recomputed with pairs of tracts weighted by a
kernel that falls to nothing at 1, 2 and 5 kilometres.

Rank correlations between the orderings of the 92 areas, each with a
probability and an interval from rearranging one of the two orderings.

The quadratic in deprivation for the share of population living on a block that
carries street commerce, with the turning point located, tested for lying
inside the range of the data, and the slope reported at each end.

Probabilities for all thirteen definitions, raw and adjusted for the number of
tests, by the method controlling the chance of any false finding and by the
method controlling the share of false findings.

Distances enter as the logarithm of one plus the distance in metres, which
keeps the tracts at zero distance in the estimates. Every estimate uses the
estimation sample marked in the tract file: the tracts that carry the
marginalisation index, a positive population and density, and a distance under
every definition, so that each definition is fitted on the same residents.
Deprivation enters on the scale standardised over that sample with population
weights, and density as the logarithm of residents per square kilometre. All
quantities are weighted by the population of the tract.

Inputs are `Results/08_tract_access.parquet`, `Results/08_area_access.parquet`,
`Results/07_block_access.parquet`, `Results/04_blocks.parquet` and
`Results/06_street_commerce.parquet`.

Outputs are `Results/11_inference.parquet`, the coefficients with raw and
adjusted probabilities, `Results/11_contrasts.parquet`, the differences between
pairs of definitions, and `Results/11_spatial.parquet`, the spatial statistics
and the standard errors allowing for spatial correlation.

Runtime is about ten minutes.

## 1. Setup

In [ ]:
import sys, os, importlib, time

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = os.environ.get('MXO_BASE', '/content/drive/MyDrive/Mexico_Obesity')
os.environ['MXO_BASE'] = PROJECT
sys.path.insert(0, f'{PROJECT}/Code')
import mxo_utils as mx
importlib.reload(mx)

assert mx.BASE == PROJECT, f'the module resolves to {mx.BASE}'

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial import cKDTree
from scipy.stats import norm, spearmanr

rep = mx.Report('11_inference').start()
mx.flow_reset()

RESULTS = mx.DIRS['results']
RNG = np.random.default_rng(20201105)
PERMUTATIONS = 999       # rearrangements for the spatial and rank statistics
NEIGHBOURS = 8           # nearest tracts of the same area
CONLEY_KM = [1.0, 2.0, 5.0]
print(f'{PERMUTATIONS} permutations, {NEIGHBOURS} nearest neighbours')

## 2. The panel

Each tract carries the distance of its median resident under every definition,
its deprivation, its density, the classification and region of its area, and
the position of its population, taken as the population weighted mean of the
coordinates of its blocks.

The estimates are made on the estimation sample marked in the tract file, so
that each definition is fitted on the same residents and the coefficients can
be compared with one another.

In [ ]:
TRACT = pd.read_parquet(os.path.join(RESULTS, '08_tract_access.parquet'))
AREA = pd.read_parquet(os.path.join(RESULTS, '08_area_access.parquet'))
BLK = pd.read_parquet(os.path.join(RESULTS, '07_block_access.parquet'),
                      columns=['CVEGEO_MZA', 'POBTOT', 'x_m', 'y_m'])
B4 = pd.read_parquet(os.path.join(RESULTS, '04_blocks.parquet'),
                     columns=['CVEGEO_MZA', 'CVEGEO'])

SETS = [col[len('median_m_'):] for col in TRACT.columns
        if col.startswith('median_m_')]
MCOL = {s: f'median_m_{s}' for s in SETS}
print(f'{len(TRACT):,} tracts, {len(SETS)} definitions')
assert len(SETS) >= 13, (
    f'the tract file carries only {len(SETS)} definitions: {SETS}. '
    'Notebook 08 has to be run again so that it picks up every set written by '
    'notebook 07.')

P = BLK.merge(B4, on='CVEGEO_MZA', how='left')
P = P[P.POBTOT > 0]
wsum = P.groupby('CVEGEO')['POBTOT'].sum()
XY = pd.DataFrame({
    'x_m': P.eval('POBTOT * x_m').groupby(P.CVEGEO).sum() / wsum,
    'y_m': P.eval('POBTOT * y_m').groupby(P.CVEGEO).sum() / wsum})

T = (TRACT.merge(AREA[['slug', 'tipo']], on='slug', how='left')
          .merge(XY, left_on='CVEGEO', right_index=True, how='left'))
T = T[T['estimation_sample'].fillna(False).astype(bool)].copy()
T['log_density'] = np.log(T['pop_density_km2'])
NEED = ['imu_depriv_z', 'log_density', 'macro_region', 'tipo', 'population',
        'slug', 'x_m', 'y_m']
assert T[NEED].notna().all().all(), 'the estimation sample lacks a characteristic'
assert T[[MCOL[s] for s in SETS]].notna().all().all(), \
    'a tract in the estimation sample lacks a distance'
T = T.reset_index(drop=True)

print(f'{len(T):,} tracts in the estimation sample, '
      f'{T.population.sum():,.0f} people')
print(f'areas {T.slug.nunique()}, tracts an area from '
      f'{T.groupby("slug").size().min()} to {T.groupby("slug").size().max()}')

for s in SETS:
    T[f'y_{s}'] = np.log1p(T[MCOL[s]])

W = T['population'].to_numpy(float)
CLUSTER = T['slug'].to_numpy()
AREAS = np.array(sorted(pd.unique(CLUSTER)))
XY_M = T[['x_m', 'y_m']].to_numpy(float)

mx.flow('tracts in the access file', len(TRACT), 'from the aggregation')
mx.flow('tracts used for inference', len(T), 'the estimation sample')

## 3. The difference between two coefficients

Every definition is regressed on deprivation and density after the population
weighted mean of its own metropolitan area has been subtracted from the outcome
and from each characteristic, which removes everything constant inside an area.

The characteristics are the same in every equation, so one score per area per
equation assembles every variance and covariance in the system. The difference
between two coefficients therefore has a standard error that accounts for the
two being estimated on the same tracts. The correlation between the two
estimates is reported with each difference, together with the standard error
the difference would carry if the two estimates were unrelated. The correlation
may be of either sign, and where it is negative the standard error of the
difference is the larger of the two.

In [ ]:
def demean(values, weight, group):
    """Each column less the weighted mean of its own group."""
    frame = pd.DataFrame(np.atleast_2d(values.T).T)
    g = pd.Series(group)
    out = np.empty_like(frame.to_numpy(float))
    den = pd.Series(weight).groupby(g).transform('sum').to_numpy()
    for j in range(frame.shape[1]):
        col = frame.iloc[:, j].to_numpy(float)
        num = pd.Series(weight * col).groupby(g).transform('sum').to_numpy()
        out[:, j] = col - num / den
    return out


XW = demean(T[['imu_depriv_z', 'log_density']].to_numpy(float), W, CLUSTER)
XtWX = XW.T @ (XW * W[:, None])
XtWX_inv = np.linalg.inv(XtWX)

beta, resid = {}, {}
for s in SETS:
    yw = demean(T[f'y_{s}'].to_numpy(float)[:, None], W, CLUSTER)[:, 0]
    b = XtWX_inv @ (XW.T @ (yw * W))
    beta[s] = b
    resid[s] = yw - XW @ b

# One score per area per equation, from which every variance and covariance in
# the system is assembled.
idx = {a: np.flatnonzero(CLUSTER == a) for a in AREAS}
scores = {s: np.array([XW[i].T @ (W[i] * resid[s][i]) for i in idx.values()])
          for s in SETS}
G, N, K = len(AREAS), len(T), XW.shape[1] + len(AREAS)
FINITE = (G / (G - 1)) * ((N - 1) / (N - K))


def covariance(a, b):
    meat = scores[a].T @ scores[b]
    return XtWX_inv @ meat @ XtWX_inv * FINITE


DEP = 0
rows = []
for s in SETS:
    v = covariance(s, s)[DEP, DEP]
    rows.append({'destination set': s, 'coefficient': beta[s][DEP],
                 'standard error': np.sqrt(v),
                 't': beta[s][DEP] / np.sqrt(v)})
COEF = pd.DataFrame(rows).set_index('destination set')
COEF['p'] = 2 * norm.sf(np.abs(COEF['t']))
print('--- coefficient on deprivation, within metropolitan areas ---\n')
print(COEF.round({'coefficient': 4, 'standard error': 4, 't': 2,
                  'p': 6}).to_string())
rep.add_table('deprivation coefficient with clustered covariance',
              COEF.reset_index(), max_rows=16)

In [ ]:
CONTRASTS = [('supermarket_club', 'fresh_perishable'),
             ('supermarket_club', 'fresh_only'),
             ('supermarket_club', 'fresh_street'),
             ('supermarket_club', 'fresh_abarrotes'),
             ('supermarket_club', 'all_food'),
             ('supermarket_mini', 'fresh_perishable'),
             ('supermarket_mini', 'fresh_only'),
             ('fresh_perishable', 'fresh_only')]
CONTRASTS = [(a, b) for a, b in CONTRASTS if a in SETS and b in SETS]

rows = []
for a, b in CONTRASTS:
    diff = beta[a][DEP] - beta[b][DEP]
    v = (covariance(a, a)[DEP, DEP] + covariance(b, b)[DEP, DEP]
         - 2 * covariance(a, b)[DEP, DEP])
    se = np.sqrt(v)
    naive = np.sqrt(covariance(a, a)[DEP, DEP] + covariance(b, b)[DEP, DEP])
    corr = covariance(a, b)[DEP, DEP] / np.sqrt(
        covariance(a, a)[DEP, DEP] * covariance(b, b)[DEP, DEP])
    rows.append({'narrower': a, 'wider': b,
                 'coefficient of the first': beta[a][DEP],
                 'coefficient of the second': beta[b][DEP],
                 'difference': diff, 'standard error': se,
                 't': diff / se, 'p': 2 * norm.sf(abs(diff / se)),
                 'correlation of the two estimates': corr,
                 'standard error if treated as independent': naive})
DIFF = pd.DataFrame(rows)
print('--- the difference between two coefficients ---\n')
print(DIFF.round({'coefficient of the first': 4,
                  'coefficient of the second': 4, 'difference': 4,
                  'standard error': 4, 't': 2, 'p': 8,
                  'correlation of the two estimates': 3,
                  'standard error if treated as independent': 4}).to_string(
    index=False))
rep.add_table('difference between the coefficients of two definitions', DIFF,
              max_rows=16)

print('\nThe final column gives the standard error the difference would '
      'carry if the\ntwo estimates were unrelated. Where the estimates are '
      'positively related the\nerror of their difference is the smaller of '
      'the two, and where they are\nnegatively related it is the larger.')

## 4. Spatial dependence

Neighbouring tracts are not independent observations. Moran's statistic
measures how far a quantity in one tract resembles the same quantity in the
tracts nearest to it. Each tract is joined to the eight nearest tracts of its
own metropolitan area, so no pair crosses a boundary between areas, and the
weights of each tract are scaled to sum to one. Values are centred on the
unweighted mean of their own area before the statistic is taken, which
confines it to resemblance between neighbours of one area. The centring is
unweighted because the statistic describes the pattern over tracts, while the
models are weighted by residents.

The statistic is computed on the access measures themselves and on the
residuals of the models of section 3, for the four definitions the argument
rests on. A value near zero means neighbouring
tracts carry unrelated values. Its probability is obtained by rearranging the
values among the tracts of each area, which holds the areas and the
neighbourhood structure fixed.

In [ ]:
rowsW, colsW = [], []
for a in AREAS:
    i = idx[a]
    if len(i) <= 1:
        continue
    k = min(NEIGHBOURS, len(i) - 1)
    tree = cKDTree(XY_M[i])
    _, nn = tree.query(XY_M[i], k=k + 1)
    for pos, neigh in enumerate(nn):
        for j in neigh[1:]:
            rowsW.append(i[pos])
            colsW.append(i[j])
rowsW, colsW = np.array(rowsW), np.array(colsW)
Wsp = sparse.csr_matrix((np.ones(len(rowsW)), (rowsW, colsW)),
                        shape=(len(T), len(T)))
deg = np.asarray(Wsp.sum(axis=1)).ravel()
Wsp = sparse.diags(np.where(deg > 0, 1 / np.maximum(deg, 1), 0.0)) @ Wsp
print(f'{Wsp.nnz:,} neighbour links, no pair crossing an area boundary')


def moran(z):
    # Values are centred within their own metropolitan area, so the statistic
    # measures resemblance between neighbouring tracts of one area and not the
    # differences between areas. Every neighbour link lies inside one area, so
    # a global centring would leave the between area differences inside the
    # statistic and inside its rearranged distribution alike.
    c = np.asarray(z, dtype=float).copy()
    for a in AREAS:
        i = idx[a]
        c[i] = c[i] - c[i].mean()
    return float(c @ (Wsp @ c) / (c @ c))


def moran_p(z, reps=PERMUTATIONS):
    observed = moran(z)
    draws = np.empty(reps)
    base = z.copy()
    for r in range(reps):
        shuffled = base.copy()
        for a in AREAS:
            i = idx[a]
            shuffled[i] = base[i][RNG.permutation(len(i))]
        draws[r] = moran(shuffled)
    greater = int((np.abs(draws) >= abs(observed)).sum())
    return observed, (greater + 1) / (reps + 1), draws.mean(), draws.std(ddof=1)


KEY = [s for s in ['supermarket_club', 'fresh_perishable', 'fresh_only',
                   'fresh_abarrotes'] if s in SETS]

t0 = time.time()
rows = []
for s in KEY:
    i_acc, p_acc, m_acc, s_acc = moran_p(T[f'y_{s}'].to_numpy(float))
    i_res, p_res, _, _ = moran_p(resid[s])
    rows.append({'destination set': s,
                 'Moran on the measure': i_acc, 'p': p_acc,
                 'Moran on the residuals': i_res, 'p of residuals': p_res,
                 'mean under rearrangement': m_acc})
MORAN = pd.DataFrame(rows).set_index('destination set')
print(f'\n--- spatial dependence, {time.time() - t0:.0f}s ---\n')
print(MORAN.round(4).to_string())
rep.add_table("Moran's statistic on the measures and the residuals",
              MORAN.reset_index(), max_rows=12)

Standard errors are recomputed with each pair of tracts weighted by a kernel
that falls linearly to nothing at a stated distance, at 1, 2 and 5 kilometres.
Only pairs inside the same metropolitan area enter.

The two kinds of standard error allow for different things. Clustering on the
area allows tracts of one area to be related at any separation and tracts of
different areas not at all. The distance weighted errors allow only pairs
inside the stated distance to be related, at a weight falling with separation.
Both are reported for each definition, with the t that each produces.

In [ ]:
def spatial_se(s, cutoff_km):
    cutoff = cutoff_km * 1000.0
    u, meat = resid[s], np.zeros((XW.shape[1], XW.shape[1]))
    for a in AREAS:
        i = idx[a]
        g = XW[i] * (W[i] * u[i])[:, None]
        tree = cKDTree(XY_M[i])
        pairs = tree.query_pairs(cutoff, output_type='ndarray')
        meat += g.T @ g                               # the diagonal terms
        if len(pairs):
            d = np.linalg.norm(XY_M[i][pairs[:, 0]] - XY_M[i][pairs[:, 1]],
                               axis=1)
            k = 1.0 - d / cutoff
            cross = (g[pairs[:, 0]] * k[:, None]).T @ g[pairs[:, 1]]
            meat += cross + cross.T
    V = XtWX_inv @ meat @ XtWX_inv * FINITE
    return float(np.sqrt(V[DEP, DEP]))


t0 = time.time()
rows = []
for s in KEY:
    r = {'destination set': s, 'coefficient': beta[s][DEP],
         'clustered on the area': COEF.loc[s, 'standard error']}
    for km in CONLEY_KM:
        r[f'correlated within {km:.0f} km'] = spatial_se(s, km)
    rows.append(r)
SPAT = pd.DataFrame(rows).set_index('destination set')
for km in CONLEY_KM:
    SPAT[f't at {km:.0f} km'] = (SPAT['coefficient']
                                 / SPAT[f'correlated within {km:.0f} km'])
print(f'--- standard errors allowing nearby tracts to be correlated, '
      f'{time.time() - t0:.0f}s ---\n')
print(SPAT.round(4).to_string())
rep.add_table('standard errors allowing for spatial correlation',
              SPAT.reset_index(), max_rows=12)

## 5. The rank correlations

The correlation between the orderings of the 92 areas under two definitions is
compared with the correlations obtained when one of the two orderings is
rearranged at random, which is the distribution it would follow if the two
orderings were unrelated.

In [ ]:
rows = []
for a, b in CONTRASTS:
    f = AREA[[MCOL[a], MCOL[b]]].dropna()
    observed = float(spearmanr(f.iloc[:, 0], f.iloc[:, 1])[0])
    first = f.iloc[:, 0].to_numpy()
    second = f.iloc[:, 1].to_numpy()
    draws = np.array([spearmanr(first, RNG.permutation(second))[0]
                      for _ in range(PERMUTATIONS)])
    greater = int((np.abs(draws) >= abs(observed)).sum())
    rows.append({'narrower': a, 'wider': b, 'areas': len(f),
                 'rank correlation': observed,
                 'p from rearrangement': (greater + 1) / (PERMUTATIONS + 1),
                 'lower': np.percentile(draws, 2.5),
                 'upper': np.percentile(draws, 97.5)})
RANK = pd.DataFrame(rows)
print('--- rank correlation between the orderings of the 92 areas ---\n')
print(RANK.round(4).to_string(index=False))
print('\nThe interval is the range the correlation would occupy if the two '
      'orderings\nwere unrelated, so an observed value outside it is not '
      'explained by chance.')
rep.add_table('rank correlation with a probability from rearrangement', RANK,
              max_rows=16)

## 6. The street commerce gradient

The share of population living on a block that carries street commerce is
regressed on deprivation, its square and density, within metropolitan areas and
weighted by population. The turning point of the quadratic is located, compared
with the range the data occupy, and the slope at each end of that range is
reported with its standard error.

In [ ]:
ST = pd.read_parquet(os.path.join(RESULTS, '06_street_commerce.parquet'),
                     columns=['CVEGEO_MZA', 'CVEGEO', 'POBTOT', 'street_any',
                              'street_known'])
ST = ST[ST.street_known].copy()
ST['pop_street'] = ST['POBTOT'].where(ST['street_any'], 0.0)
SH = ST.groupby('CVEGEO').agg(population=('POBTOT', 'sum'),
                              on_street=('pop_street', 'sum'))
SH['share'] = SH.on_street / SH.population.replace(0, np.nan)
S = T.merge(SH[['share']], left_on='CVEGEO', right_index=True, how='inner')
S = S.dropna(subset=['share'])
S = S[S.population > 0]
print(f'{len(S):,} tracts with a street commerce share, '
      f'{S.population.sum():,.0f} people')

ws = S.population.to_numpy(float)
cl = S.slug.to_numpy()
dep = S.imu_depriv_z.to_numpy(float)
Xs = demean(np.column_stack([dep, dep ** 2,
                             S.log_density.to_numpy(float)]), ws, cl)
ys = demean(S.share.to_numpy(float)[:, None], ws, cl)[:, 0]
A = Xs.T @ (Xs * ws[:, None])
bs = np.linalg.solve(A, Xs.T @ (ys * ws))
us = ys - Xs @ bs
Ainv = np.linalg.inv(A)
mt = np.zeros((3, 3))
for a in pd.unique(cl):
    i = np.flatnonzero(cl == a)
    sc = Xs[i].T @ (ws[i] * us[i])
    mt += np.outer(sc, sc)
Gs = len(pd.unique(cl))
Vs = Ainv @ mt @ Ainv * (Gs / (Gs - 1)) * ((len(S) - 1) / (len(S) - 3 - Gs))
se = np.sqrt(np.diag(Vs))

print('\n--- the quadratic in deprivation ---\n')
QUAD = pd.DataFrame({'term': ['deprivation', 'deprivation squared',
                              'log density'],
                     'coefficient': bs, 'standard error': se,
                     't': bs / se, 'p': 2 * norm.sf(np.abs(bs / se))})
print(QUAD.round({'coefficient': 5, 'standard error': 5, 't': 2,
                  'p': 6}).to_string(index=False))

turn = -bs[0] / (2 * bs[1])
lo, hi = float(dep.min()), float(dep.max())
slope_lo, slope_hi = bs[0] + 2 * bs[1] * lo, bs[0] + 2 * bs[1] * hi
grad = lambda x: np.array([1.0, 2 * x, 0.0])
se_lo = float(np.sqrt(grad(lo)[:2] @ Vs[:2, :2] @ grad(lo)[:2]))
se_hi = float(np.sqrt(grad(hi)[:2] @ Vs[:2, :2] @ grad(hi)[:2]))
print(f'\nturning point at deprivation {turn:.3f}, the data running from '
      f'{lo:.3f} to {hi:.3f}')
print(f'inside the range of the data: {lo < turn < hi}')
print(f'slope at the least deprived end  {slope_lo:+.5f} '
      f'(se {se_lo:.5f}, t {slope_lo / se_lo:+.2f})')
print(f'slope at the most deprived end   {slope_hi:+.5f} '
      f'(se {se_hi:.5f}, t {slope_hi / se_hi:+.2f})')
print('\nA curve requires the turning point inside the range and the two '
      'slopes\nopposite in sign, each distinguishable from zero.')
rep.add_table('the quadratic in deprivation for street commerce', QUAD,
              max_rows=10)
UQ = pd.DataFrame([{'turning point': round(turn, 4),
                    'lowest deprivation': round(lo, 4),
                    'highest deprivation': round(hi, 4),
                    'inside the range': bool(lo < turn < hi),
                    'slope at the low end': round(slope_lo, 5),
                    'se low': round(se_lo, 5),
                    'slope at the high end': round(slope_hi, 5),
                    'se high': round(se_hi, 5)}])
rep.add_table('the shape of the street commerce gradient', UQ, max_rows=10)

## 7. Thirteen tests at once

Probabilities are reported as they come and after two adjustments across the
thirteen definitions, one controlling the chance of any false finding and one
controlling the share of false findings among those declared.

In [ ]:
def holm(p):
    order = np.argsort(p)
    m = len(p)
    adjusted = np.empty(m)
    running = 0.0
    for rank, j in enumerate(order):
        running = max(running, (m - rank) * p[j])
        adjusted[j] = min(running, 1.0)
    return adjusted


def benjamini_hochberg(p):
    order = np.argsort(p)
    m = len(p)
    adjusted = np.empty(m)
    running = 1.0
    for rank in range(m - 1, -1, -1):
        j = order[rank]
        running = min(running, m / (rank + 1) * p[j])
        adjusted[j] = min(running, 1.0)
    return adjusted


ADJ = COEF[['coefficient', 'standard error', 't', 'p']].copy()
ADJ['p adjusted, any false finding'] = holm(COEF['p'].to_numpy())
ADJ['p adjusted, share of false findings'] = benjamini_hochberg(
    COEF['p'].to_numpy())
print(ADJ.round({'coefficient': 4, 'standard error': 4, 't': 2, 'p': 8,
                 'p adjusted, any false finding': 8,
                 'p adjusted, share of false findings': 8}).to_string())
rep.add_table('probabilities adjusted for the number of definitions tested',
              ADJ.reset_index(), max_rows=16)

survive = int((ADJ['p adjusted, any false finding'] < 0.05).sum())
print(f'\ndefinitions whose coefficient remains distinguishable from zero '
      f'after adjustment: {survive} of {len(SETS)}')

## 8. Checks

Each check names the condition under which it fails.

In [ ]:
print('--- checks ---\n')

d = int(T['CVEGEO'].duplicated().sum())
print(f'1. duplicate tracts                          {d}')
assert d == 0

# Removing what is constant inside an area must leave no mean inside an area.
left = np.max([abs((W[i] * XW[i, 0]).sum() / W[i].sum()) for i in idx.values()])
print(f'2. largest group mean left after demeaning   {left:.2e}')
assert left < 1e-8

# The variance of the difference has to equal the two variances less twice the
# covariance. Where the covariance is negative the difference is the less
# precisely estimated of the two, which is admissible.
rebuilt = []
for a, b in CONTRASTS:
    rebuilt.append(np.sqrt(covariance(a, a)[DEP, DEP]
                           + covariance(b, b)[DEP, DEP]
                           - 2 * covariance(a, b)[DEP, DEP]))
gap = float(np.max(np.abs(np.array(rebuilt) - DIFF['standard error'].to_numpy())))
print(f'3. the identity for the variance of a difference, largest gap {gap:.2e}')
assert gap < 1e-12

neg = int((DIFF['correlation of the two estimates'] < 0).sum())
print(f'   of {len(DIFF)} contrasts, {neg} have negatively related estimates '
      f'and so a larger error than independence implies')

# Every pair of estimates must carry a covariance matrix that is admissible.
bad = 0
for a, b in CONTRASTS:
    M = np.array([[covariance(a, a)[DEP, DEP], covariance(a, b)[DEP, DEP]],
                  [covariance(a, b)[DEP, DEP], covariance(b, b)[DEP, DEP]]])
    if np.min(np.linalg.eigvalsh(M)) < -1e-18:
        bad += 1
print(f'4. pairs whose covariance matrix is not admissible  {bad}')
assert bad == 0

assert DIFF['correlation of the two estimates'].between(-1, 1).all()
print('   every correlation between estimates inside minus one to one  yes')

# Moran's statistic on row standardised weights lies between minus one and one.
assert MORAN['Moran on the measure'].between(-1, 1).all()
assert MORAN['Moran on the residuals'].between(-1, 1).all()
print('5. Moran statistics inside minus one to one  yes')

# Rearranging the values must centre the statistic near zero.
print(f'6. largest mean under rearrangement          '
      f'{MORAN["mean under rearrangement"].abs().max():.4f}')
assert MORAN['mean under rearrangement'].abs().max() < 0.05

# An adjusted probability can only be larger than the one it adjusts.
assert (ADJ['p adjusted, any false finding'] >= ADJ['p'] - 1e-12).all()
assert (ADJ['p adjusted, share of false findings'] >= ADJ['p'] - 1e-12).all()
print('7. adjusted probabilities never smaller than the raw ones  yes')

# Every weight row must sum to one or the tract has no neighbour.
sums = np.asarray(Wsp.sum(axis=1)).ravel()
print(f'8. tracts whose neighbour weights do not sum to one  '
      f'{int((np.abs(sums - 1) > 1e-9).sum())}')

key = DIFF[(DIFF.narrower == 'supermarket_club')
           & (DIFF.wider == 'fresh_only')]
if len(key):
    r = key.iloc[0]
    print(f'\n9. supermarkets against specialist fresh, difference '
          f'{r["difference"]:+.4f}, se {r["standard error"]:.4f}, '
          f't {r["t"]:+.2f}, p {r["p"]:.2e}')
print(f'10. tracts used                              {len(T):,}')

# Deprivation has weighted mean zero and unit weighted spread on the sample.
wm = float((W * T.imu_depriv_z).sum() / W.sum())
ws = float(np.sqrt((W * (T.imu_depriv_z - wm) ** 2).sum() / W.sum()))
print(f'12. deprivation, weighted mean {wm:.2e} and weighted sd {ws:.6f}')
assert abs(wm) < 1e-6 and abs(ws - 1) < 1e-6
print(f'11. areas used                               {len(AREAS)}')

## 9. Write results

In [ ]:
out = os.path.join(RESULTS, '11_inference.parquet')
COEF.reset_index().merge(
    ADJ.reset_index()[['destination set',
                       'p adjusted, any false finding',
                       'p adjusted, share of false findings']],
    on='destination set', how='left').to_parquet(out, index=False)
print('written:', out)

outd = os.path.join(RESULTS, '11_contrasts.parquet')
DIFF.to_parquet(outd, index=False)
print('written:', outd)

outs = os.path.join(RESULTS, '11_spatial.parquet')
MORAN.join(SPAT, rsuffix=' spatial').reset_index().to_parquet(outs,
                                                              index=False)
print('written:', outs)

print()
print(mx.flow_table().to_string(index=False))
mx.save_flow(os.path.join(mx.DIRS['logs'], '11_sample_flow.csv'))
rep.add_table('attrition', mx.flow_table(), max_rows=20)

if len(key):
    r = key.iloc[0]
    rep.note(f'the difference between the deprivation coefficients of '
             f'supermarkets with clubs and specialist fresh sellers is '
             f'{r["difference"]:+.4f} with a standard error of '
             f'{r["standard error"]:.4f}, t {r["t"]:+.2f}, p {r["p"]:.2e}; '
             f"Moran's statistic on the residuals is "
             f'{MORAN.loc["fresh_only", "Moran on the residuals"]:.3f}; '
             f'{survive} of {len(SETS)} coefficients survive adjustment for '
             f'multiple testing')
rep.save()